# Free-Uncensored-Qwen38-Pro (نسخه پایدار)

**اجرای مدل Uncensored Qwen3.8-27B روی Colab رایگان**

الهام گرفته از پروژه MorTsaedi.

### ترتیب اجرا
1. Runtime → Change runtime type → **T4 GPU** را انتخاب و Save کنید.
2. سلول ۱ را اجرا کنید (رمز را عوض کنید).
3. سلول ۲ را اجرا کنید (۵-۱۵ دقیقه).
4. سلول ۳ را اجرا کنید.
5. دستورات SSH را کپی کنید.

تب را باز نگه دارید.

In [ ]:
# @title ۱ - تنظیمات

PASSWORD = "MyStrongPass123!"  # رمز را عوض کنید
HF_TOKEN = ""  # اختیاری

MODEL_REPO = "OBLITERATUS/Qwen3.8-27B-OBLITERATED"
MODEL_QUANT = "Q3_K_M"
MODEL_FILE = f"Qwen3.8-27B-OBLITERATED-{MODEL_QUANT}.gguf"
MODEL_DIR = "/content/models"
MODEL_PATH = f"{MODEL_DIR}/{MODEL_FILE}"
SERVER_PORT = 3005
CTX_SIZE = 32768
N_GPU_LAYERS = 99

print("✅ تنظیمات آماده شد")
print(f"مدل: {MODEL_REPO} | Quant: {MODEL_QUANT}")
print(f"رمز: {PASSWORD}")

In [ ]:
# @title ۲ - نصب و دانلود مدل

import os
from pathlib import Path

print("در حال آماده‌سازی...")
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq openssh-server git build-essential cmake > /dev/null 2>&1
!pip install -q huggingface_hub > /dev/null 2>&1

os.makedirs(MODEL_DIR, exist_ok=True)

LLAMA_DIR = "/content/llama.cpp"
if not Path(f"{LLAMA_DIR}/llama-server").exists():
    print("کلون و ساخت llama.cpp...")
    !rm -rf {LLAMA_DIR}
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git {LLAMA_DIR} > /dev/null 2>&1
    %cd {LLAMA_DIR}
    !cmake -B build -DGGML_CUDA=ON > /dev/null 2>&1 || cmake -B build > /dev/null 2>&1
    !cmake --build build --config Release -j$(nproc) --target llama-server > /dev/null 2>&1
    %cd /content
    if Path(f"{LLAMA_DIR}/build/bin/llama-server").exists():
        !cp {LLAMA_DIR}/build/bin/llama-server {LLAMA_DIR}/llama-server
        print("✅ llama-server آماده")
    else:
        %cd {LLAMA_DIR}
        !make -j$(nproc) llama-server > /dev/null 2>&1
        %cd /content
else:
    print("llama-server از قبل موجود است")

print("دانلود مدل...")
from huggingface_hub import hf_hub_download
try:
    path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir=MODEL_DIR, token=HF_TOKEN or None, resume_download=True)
    print("✅ مدل دانلود شد:", path)
except Exception as e:
    print("خطا:", e)
    print("توکن HF را در سلول ۱ وارد کنید: https://huggingface.co/settings/tokens")
    raise
print("✅ آماده. سلول ۳ را اجرا کنید.")

In [ ]:
# @title ۳ - راه‌اندازی سرور و تونل

import subprocess, threading, time, re
from pathlib import Path

print("شروع راه‌اندازی...")

!mkdir -p /var/run/sshd
!echo "root:{PASSWORD}" | chpasswd
!sed -i 's/#*PermitRootLogin.*/PermitRootLogin yes/' /etc/ssh/sshd_config
!sed -i 's/#*PasswordAuthentication.*/PasswordAuthentication yes/' /etc/ssh/sshd_config
!service ssh restart > /dev/null 2>&1
print("✅ SSH آماده")

server_bin = "/content/llama.cpp/llama-server"
if not Path(server_bin).exists():
    server_bin = "/content/llama.cpp/build/bin/llama-server"

if not Path(server_bin).exists():
    print("❌ llama-server پیدا نشد. سلول ۲ را دوباره اجرا کنید.")
else:
    cmd = [server_bin, "-m", MODEL_PATH, "--host", "127.0.0.1", "--port", str(SERVER_PORT), "-c", str(CTX_SIZE), "-ngl", str(N_GPU_LAYERS), "--jinja"]
    print("شروع llama-server...")
    server_proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(15)
    print("✅ llama-server در حال اجرا")

    print("راه‌اندازی bore...")
    !curl -sL https://github.com/ekzhang/bore/releases/download/v0.5.2/bore-v0.5.2-x86_64-unknown-linux-musl.tar.gz | tar xz -C /tmp 2>/dev/null || true
    if not Path("/tmp/bore").exists():
        !curl -sL https://github.com/ekzhang/bore/releases/latest/download/bore-x86_64-unknown-linux-musl.tar.gz | tar xz -C /tmp 2>/dev/null || true

    bore_proc = subprocess.Popen(["/tmp/bore", "local", "22", "--to", "bore.pub"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    bore_port = None
    for _ in range(50):
        line = bore_proc.stdout.readline()
        if line:
            print(line.strip())
            m = re.search(r"bore\.pub:(\d+)", line)
            if m:
                bore_port = m.group(1)
                break
        time.sleep(0.3)

    if bore_port:
        print("\n" + "="*50)
        print("✅ سرور آماده شد!")
        print("="*50)
        print(f"پورت: {bore_port}")
        print(f"رمز: {PASSWORD}")
        print("\nدستور SSH:")
        print(f"ssh -o StrictHostKeyChecking=no -o ServerAliveInterval=30 -p {bore_port} root@bore.pub")
        print("\nForward (ترمینال جدا):")
        print(f"ssh -N -o StrictHostKeyChecking=no -o ServerAliveInterval=30 -p {bore_port} -L {SERVER_PORT}:127.0.0.1:{SERVER_PORT} root@bore.pub")
        print(f"\nبعد از Forward: http://localhost:{SERVER_PORT}")
        print("="*50)
    else:
        print("❌ پورت bore گرفته نشد. دوباره سلول ۳ را اجرا کنید.")

    def keep_alive():
        while True:
            time.sleep(40)
            print("·", end="", flush=True)
    threading.Thread(target=keep_alive, daemon=True).start()
    print("\n💚 Keep-alive فعال. تب را باز نگه دارید.")

In [ ]:
# @title ۴ - تست API (اختیاری)

import requests, time
print("تست API...")
time.sleep(3)
try:
    r = requests.post(f"http://127.0.0.1:{SERVER_PORT}/v1/chat/completions", json={"model":"local","messages":[{"role":"user","content":"بگو آماده‌ای"}],"max_tokens":20}, timeout=90)
    if r.status_code == 200:
        print("✅ API کار می‌کند!")
        print(r.json()["choices"][0]["message"]["content"])
    else:
        print(r.status_code, r.text[:200])
except Exception as e:
    print("هنوز آماده نیست:", str(e)[:150])
    print("۳۰ ثانیه صبر کنید و دوباره اجرا کنید.")

---

### راهنما

1. دستور SSH را اجرا کنید.
2. دستور Forward را در ترمینال جدا اجرا کنید و باز نگه دارید.
3. بروید به http://localhost:3005

اگر قطع شد، سلول ۳ را دوباره اجرا کنید.

**Attribution:** الهام گرفته از [MorTsaedi](https://github.com/MorTsaedi/Free-Uncensored-Qwen3.8-27B)